<div align="center">

# 🧠 **CBS5502 — Computational Linguistics and NLP Technologies**

### 🐍 **Python Tutorial**
### 📅 *March 4, 2026*

---

## 🏷️ **Named Entity Recognition (NER)**
## 🔁 **Sequence Labelling and Ambiguity**

---

### 👨‍🏫 **Instructor**  
**Dr. WAN Mingyu**

### 👨‍🏫 **Teaching Assistant**  
**Mr. BAO Xiaoyi**

</div>

---
## 🎯 Learning Objectives

By the end of this tutorial, you will be able to:

- Understand NER as a **sequence labelling task**
- Apply the **BIO tagging scheme**
- Recognize and analyze **entity ambiguity**
- Implement multiple NER models:
  - Baseline model
  - Context‑free classifier
  - BiLSTM sequence model
  - BiLSTM + CRF structured model
- Evaluate performance using **entity‑level F1**
- Interpret model behavior on ambiguous cases

---

## 🧠 What is Named Entity Recognition?

Named Entity Recognition (NER) is the task of:

> ✅ Identifying spans of text  
> ✅ Classifying them into predefined categories  

Common entity types include:

- 👤 PERSON
- 🏢 ORGANIZATION
- 🌍 LOCATION
- 📅 DATE
- 📦 PRODUCT

---


In [ ]:
!pip install torch seqeval torchcrf

## Why These Libraries?

- torch → Neural network implementation
- seqeval → Entity-level evaluation (strict BIO scoring)
- pytorch-crf → Conditional Random Field layer

---

## ⚠️ Why NER is Challenging

NER must handle **contextual ambiguity**.

Consider:

| Sentence | Interpretation |
|-----------|----------------|
| Apple released a new product. | Apple → ORG |
| Apple fell from the tree. | Apple → O |
| Jordan won the match. | Jordan → PER |
| Jordan signed a treaty. | Jordan → LOC |
| Washington passed the bill. | Washington → PER |
| Washington is rainy in winter. | Washington → LOC |

The same word can represent:

- A person
- A location
- An organization
- Or nothing at all

Correct tagging requires **context modelling**.

---

## 🧩 What We Will Do in This Tutorial

1️⃣ Construct an ambiguity‑focused dataset  
2️⃣ Encode BIO labels  
3️⃣ Train multiple NER models  
4️⃣ Compare performance across methods  
5️⃣ Analyze model behavior  
6️⃣ Understand why structured modelling improves results  

---

## ✅ Key Question

How much does:

- Context
- Sequence modelling
- Structured prediction

actually improve NER performance?

We will find out experimentally.

---

# 🚀 Let’s Begin!

In [ ]:
data = [

# ---------------------------
# Apple
# ---------------------------

(["Apple", "released", "a", "new", "MacBook"],
 ["B-ORG", "O", "O", "O", "B-PROD"]),

(["Apple", "fell", "from", "the", "tree"],
 ["O", "O", "O", "O", "O"]),

(["Tim", "Cook", "leads", "Apple"],
 ["B-PER", "I-PER", "O", "B-ORG"]),

(["Apple", "is", "headquartered", "in", "California"],
 ["B-ORG", "O", "O", "O", "B-LOC"]),

# ---------------------------
# Jordan
# ---------------------------

(["Jordan", "won", "the", "match"],
 ["B-PER", "O", "O", "O"]),

(["Jordan", "signed", "a", "peace", "agreement"],
 ["B-LOC", "O", "O", "O", "O"]),

(["Michael", "Jordan", "is", "a", "legend"],
 ["B-PER", "I-PER", "O", "O", "O"]),

(["Jordan", "is", "located", "in", "the", "Middle", "East"],
 ["B-LOC", "O", "O", "O", "O", "B-LOC", "I-LOC"]),

# ---------------------------
# Washington
# ---------------------------

(["Washington", "passed", "the", "law"],
 ["B-PER", "O", "O", "O"]),

(["Washington", "is", "rainy", "in", "winter"],
 ["B-LOC", "O", "O", "O", "O"]),

(["George", "Washington", "was", "the", "first", "president"],
 ["B-PER", "I-PER", "O", "O", "O", "O"]),

(["The", "Washington", "Post", "published", "an", "article"],
 ["O", "B-ORG", "I-ORG", "O", "O", "O"]),

# ---------------------------
# Amazon
# ---------------------------

(["Amazon", "reported", "strong", "profits"],
 ["B-ORG", "O", "O", "O"]),

(["The", "Amazon", "flows", "through", "Brazil"],
 ["O", "B-LOC", "O", "O", "B-LOC"]),

(["Jeff", "Bezos", "founded", "Amazon"],
 ["B-PER", "I-PER", "O", "B-ORG"]),

(["Amazon", "expanded", "its", "delivery", "service"],
 ["B-ORG", "O", "O", "O", "O"]),

# ---------------------------
# Paris
# ---------------------------

(["Paris", "hosted", "the", "Olympics"],
 ["B-LOC", "O", "O", "O"]),

(["Paris", "Hilton", "attended", "the", "event"],
 ["B-PER", "I-PER", "O", "O", "O"]),

(["The", "conference", "was", "held", "in", "Paris"],
 ["O", "O", "O", "O", "O", "B-LOC"]),

# ---------------------------
# Cambridge
# ---------------------------

(["Cambridge", "is", "a", "beautiful", "city"],
 ["B-LOC", "O", "O", "O", "O"]),

(["Cambridge", "Analytica", "faced", "controversy"],
 ["B-ORG", "I-ORG", "O", "O"]),

(["She", "studied", "at", "the", "University", "of", "Cambridge"],
 ["O", "O", "O", "O", "B-ORG", "I-ORG", "I-ORG"]),

# ---------------------------
# Ford
# ---------------------------

(["Ford", "introduced", "a", "new", "SUV"],
 ["B-ORG", "O", "O", "O", "O"]),

(["The", "Ford", "River", "is", "polluted"],
 ["O", "B-LOC", "I-LOC", "O", "O"]),

(["Henry", "Ford", "founded", "Ford"],
 ["B-PER", "I-PER", "O", "B-ORG"]),

# ---------------------------
# Tesla
# ---------------------------

(["Tesla", "announced", "a", "new", "battery"],
 ["B-ORG", "O", "O", "O", "O"]),

(["Nikola", "Tesla", "invented", "AC", "power"],
 ["B-PER", "I-PER", "O", "O", "O"]),

(["Tesla", "shares", "rose", "today"],
 ["B-ORG", "O", "O", "O"]),

# ---------------------------
# Oracle
# ---------------------------

(["Oracle", "acquired", "a", "startup"],
 ["B-ORG", "O", "O", "O"]),

(["The", "Oracle", "predicted", "the", "future"],
 ["O", "O", "O", "O", "O"]),

# ---------------------------
# Multi-word boundary cases
# ---------------------------

(["New", "York", "City", "is", "crowded"],
 ["B-LOC", "I-LOC", "I-LOC", "O", "O"]),

(["The", "New", "York", "Times", "published", "a", "report"],
 ["O", "B-ORG", "I-ORG", "I-ORG", "O", "O", "O"]),

(["Bank", "of", "America", "announced", "profits"],
 ["B-ORG", "I-ORG", "I-ORG", "O", "O"]),

(["University", "of", "California", "Berkeley", "is", "prestigious"],
 ["B-ORG", "I-ORG", "I-ORG", "I-ORG", "O", "O"]),

(["California", "is", "sunny"],
 ["B-LOC", "O", "O"]),

(["Facebook", "hired", "engineers"],
 ["B-ORG", "O", "O"]),

(["Meta", "owns", "Facebook"],
 ["B-ORG", "O", "B-ORG"]),

]

# 🟢 Model 1 — Baseline Approach

A baseline model:

- Establishes a performance reference point  
- Reveals dataset difficulty  
- Highlights the importance of modelling decisions  
- Prevents overestimating improvements  

In NER/NED, a naive strategy already tells us something important about the task.

---

# 🎯 Baseline Strategy

### Majority Class Baseline

This model predicts:

> Every token → `O` (Outside any entity)

In other words:

- It assumes **no entities exist**
- It ignores context
- It ignores sequence structure
- It ignores ambiguity

---

## 🧠 Why Is This a Valid Baseline?

Because in most corpora:

- The majority of tokens are `O`
- Entities are sparse relative to total tokens

Therefore:

- Token-level accuracy may appear high
- Entity-level F1 will be very low

This exposes the **evaluation paradox** in NER.

# 🔍 Why This Matters for NED

In Named Entity Detection:

- Most words are not entities
- Simply predicting `O` often yields:
  - High token accuracy
  - Near-zero entity recall

This demonstrates why:

> ✅ Entity-level evaluation is essential.






In [ ]:
import torch
from sklearn.model_selection import train_test_split

# Split dataset
train_data, test_data = train_test_split(data, test_size=0.3, random_state=42)

# Build vocabulary
vocab = {"<PAD>":0, "<UNK>":1}
idx = 2

for tokens, _ in train_data:
    for word in tokens:
        if word not in vocab:
            vocab[word] = idx
            idx += 1

# Label mapping
labels = sorted(list(set(l for _, y in data for l in y)))
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

In [ ]:
def encode_dataset(dataset):
    X, Y = [], []
    for tokens, tags in dataset:
        X.append(torch.tensor([vocab.get(w,1) for w in tokens]))
        Y.append(torch.tensor([label2id[t] for t in tags]))
    return X, Y

X_train, Y_train = encode_dataset(train_data)
X_test, Y_test = encode_dataset(test_data)

In [ ]:
from seqeval.metrics import classification_report, f1_score

true_labels = []
pred_labels = []

for y in Y_test:
    gold = [id2label[i.item()] for i in y]
    pred = ["O"] * len(gold)
    true_labels.append(gold)
    pred_labels.append(pred)

print("Majority Baseline")
print(classification_report(true_labels, pred_labels))
print("F1:", f1_score(true_labels, pred_labels))

Majority Baseline
              precision    recall  f1-score   support

         LOC       0.00      0.00      0.00         3
         ORG       0.00      0.00      0.00         4
         PER       0.00      0.00      0.00         6

   micro avg       0.00      0.00      0.00        13
   macro avg       0.00      0.00      0.00        13
weighted avg       0.00      0.00      0.00        13

F1: 0.0


# 🟡 Model 2 — Context‑Free Token Classifier

---

## 🔎 Moving Beyond the Baseline

Now we introduce a slightly more intelligent model:

> A token-level classifier that learns from data.

However, this model still treats each word **independently**.

It does not model:

- Sentence structure  
- Label transitions  
- Global sequence dependencies  

---
For each token:

[
y_i = \arg\max P(y_i \mid x_i)
]

The prediction depends only on:

- The word itself
- Its learned embedding

Not on:

- Neighboring words
- Previous labels
- Entire sentence context

---

# 🧠 What Does “Context‑Free” Mean?

Context-free here means:

- The model ignores surrounding tokens
- Each word is classified in isolation
- Sequence structure is not considered

Formally:

[
P(y \mid x) \approx \prod_i P(y_i \mid x_i)
]

No modelling of:

[
P(y_i \mid x, y_{i-1})
]

---

# 📊 What Does This Model Capture?

| Component | Included? |
|------------|------------|
| Word Identity | ✅ |
| Learned Word Representation | ✅ |
| Context | ❌ |
| Label Transitions | ❌ |
| BIO Validity Constraints | ❌ |
| Ambiguity Resolution via Context | ❌ |

---

# 🔍 Why Is This Stronger Than Baseline?

Unlike the baseline:

- It learns that "Apple" is often B-ORG
- It learns that "Jordan" is often B-PER
- It captures lexical statistics

This introduces:

> ✅ Data-driven prediction

---


In [ ]:
import torch.nn as nn

class TokenClassifier(nn.Module):
    def __init__(self, vocab_size, tagset_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64)
        self.fc = nn.Linear(64, tagset_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.fc(x)
        return x

model_token = TokenClassifier(len(vocab), len(label2id))

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_token.parameters(), lr=0.01)

for epoch in range(30):
    for x,y in zip(X_train, Y_train):
        x = x.unsqueeze(0)
        y = y.unsqueeze(0)

        optimizer.zero_grad()
        outputs = model_token(x)
        loss = loss_fn(outputs.view(-1, len(label2id)), y.view(-1))
        loss.backward()
        optimizer.step()

In [ ]:
model_token.eval()

true_labels = []
pred_labels = []

for x,y in zip(X_test, Y_test):
    x = x.unsqueeze(0)
    outputs = model_token(x)
    preds = torch.argmax(outputs, dim=2)

    pred = [id2label[i.item()] for i in preds[0]]
    gold = [id2label[i.item()] for i in y]

    true_labels.append(gold)
    pred_labels.append(pred)

print("Token Classifier")
print(classification_report(true_labels, pred_labels))
print("F1:", f1_score(true_labels, pred_labels))

Token Classifier
              precision    recall  f1-score   support

         LOC       0.00      0.00      0.00         3
         ORG       0.50      0.75      0.60         4
         PER       1.00      0.17      0.29         6

   micro avg       0.14      0.31      0.19        13
   macro avg       0.50      0.31      0.30        13
weighted avg       0.62      0.31      0.32        13

F1: 0.1904761904761905


# 🔵 Model 3 — BiLSTM Sequence Model

---

## 🔎 From Independent Tokens to Contextual Sequences

The previous model treated each word **independently**:

[
P(y_i \mid x_i)
]

But language is inherently sequential.

Meaning depends on:

- Words before
- Words after
- Grammatical structure
- Contextual cues

To model this, we introduce a **Bidirectional LSTM (BiLSTM)**.

For each token:

[
h_i = BiLSTM(x_1, x_2, ..., x_n)
]

[
y_i = \arg\max P(y_i \mid h_i)
]

Each prediction now depends on:

✅ The entire sentence  
✅ Both left and right context  
✅ Learned contextual representations  

---

# 🧠 Why Bidirectional?

A standard LSTM reads:

- Left → Right

But NER often requires:

- Future context
- Sentence-level disambiguation



In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, tagset_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64)
        self.lstm = nn.LSTM(64, 64, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(128, tagset_size)

    def forward(self, x):
        x = self.embedding(x)
        x,_ = self.lstm(x)
        x = self.fc(x)
        return x

model_lstm = BiLSTM(len(vocab), len(label2id))

In [ ]:
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=0.01)

for epoch in range(40):
    for x,y in zip(X_train, Y_train):
        x = x.unsqueeze(0)
        y = y.unsqueeze(0)

        optimizer.zero_grad()
        outputs = model_lstm(x)
        loss = loss_fn(outputs.view(-1, len(label2id)), y.view(-1))
        loss.backward()
        optimizer.step()

In [ ]:
model_lstm.eval()

true_labels = []
pred_labels = []

for x,y in zip(X_test, Y_test):
    x = x.unsqueeze(0)
    outputs = model_lstm(x)
    preds = torch.argmax(outputs, dim=2)

    pred = [id2label[i.item()] for i in preds[0]]
    gold = [id2label[i.item()] for i in y]

    true_labels.append(gold)
    pred_labels.append(pred)

print("BiLSTM")
print(classification_report(true_labels, pred_labels))
print("F1:", f1_score(true_labels, pred_labels))

BiLSTM
              precision    recall  f1-score   support

         LOC       0.14      0.33      0.20         3
         ORG       0.12      0.25      0.17         4
         PER       0.00      0.00      0.00         6

   micro avg       0.13      0.15      0.14        13
   macro avg       0.09      0.19      0.12        13
weighted avg       0.07      0.15      0.10        13

F1: 0.14285714285714288


 # 🔴 Model 4 — BiLSTM + CRF Structured Model

---

## 🔎 From Context to Structure

The BiLSTM model successfully captured:

✅ Left context  
✅ Right context  
✅ Contextual semantics  
✅ Many ambiguity cases  

However, it still made predictions **independently per token**.

NER is not only contextual — it is also **structurally constrained**.

This motivates the addition of a:

> 🔗 Conditional Random Field (CRF) layer.

---
The BiLSTM provides:

- Contextual token representations

The CRF provides:

- Sequence-level structured decoding

Instead of predicting each label independently, the model finds:

[
y^* = \arg\max_y Score(x, y)
]

for the entire sequence.

---

# 🧠 What Does the CRF Add?

The CRF models:

1️⃣ Emission Scores  
2️⃣ Transition Scores  

The total sequence score:

[
Score(x, y) =
\sum_i Emission_i(y_i)
+
\sum_i Transition(y_{i-1}, y_i)
]

Where:

- Emission_i comes from BiLSTM output
- Transition scores are learned parameters

---



In [ ]:
!pip install pytorch-crf

In [ ]:
from torchcrf import CRF

class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, tagset_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64)
        self.lstm = nn.LSTM(64, 64, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(128, tagset_size)
        self.crf = CRF(tagset_size, batch_first=True)

    def forward(self, x, tags=None):
        mask = x != 0
        x = self.embedding(x)
        x,_ = self.lstm(x)
        emissions = self.fc(x)

        if tags is not None:
            return -self.crf(emissions, tags, mask=mask)
        else:
            return self.crf.decode(emissions, mask=mask)

model_crf = BiLSTM_CRF(len(vocab), len(label2id))

In [ ]:
optimizer = torch.optim.Adam(model_crf.parameters(), lr=0.01)

for epoch in range(40):
    for x,y in zip(X_train, Y_train):
        x = x.unsqueeze(0)
        y = y.unsqueeze(0)

        optimizer.zero_grad()
        loss = model_crf(x,y)
        loss.backward()
        optimizer.step()

In [ ]:
model_crf.eval()

true_labels = []
pred_labels = []

for x,y in zip(X_test, Y_test):
    x = x.unsqueeze(0)
    preds = model_crf(x)

    pred = [id2label[i] for i in preds[0]]
    gold = [id2label[i.item()] for i in y]

    true_labels.append(gold)
    pred_labels.append(pred)

print("BiLSTM + CRF")
print(classification_report(true_labels, pred_labels))
print("F1:", f1_score(true_labels, pred_labels))

BiLSTM + CRF
              precision    recall  f1-score   support

         LOC       0.14      0.33      0.20         3
         ORG       0.60      0.75      0.67         4
         PER       1.00      0.17      0.29         6

   micro avg       0.38      0.38      0.38        13
   macro avg       0.58      0.42      0.38        13
weighted avg       0.68      0.38      0.38        13

F1: 0.38461538461538464


# 📊 Model Comparison & Performance Discussion

After training multiple models, we now compare their behavior and performance.

We experimented with:

1️⃣ Majority Baseline  
2️⃣ Token-Level Classifier (No Context)  
3️⃣ BiLSTM Sequence Model  
4️⃣ BiLSTM + CRF Structured Model  

---

## 🔢 Performance Overview

Typical performance pattern:

| Model | Context Awareness | Sequence Modelling | Expected F1 |
|--------|------------------|-------------------|-------------|
| Majority Baseline | ❌ | ❌ | Very Low |
| Token Classifier | ❌ | ❌ | Low |
| BiLSTM | ✅ | Partial | Medium |
| BiLSTM + CRF | ✅ | ✅ | Highest |

---

# 🧠 Why Do Performances Differ?

## 1️⃣ Majority Baseline

- Always predicts `O`
- Ignores entity structure entirely
- High token accuracy but near-zero entity F1

### ✅ Lesson:
Token-level accuracy is misleading.  
Entity-level evaluation is essential.

---

## 2️⃣ Token-Level Classifier (No Context)

Architecture:

Characteristics:

- Each word treated independently
- No access to neighboring words
- Same surface word always gets similar prediction

### ⚠️ Ambiguity Failure

Example:

| Sentence | Correct Label | Likely Error |
|-----------|--------------|--------------|
| Apple released iOS | B-ORG | ✅ |
| Apple fell from tree | O | ❌ Often predicted ORG |

Without context, the model cannot disambiguate meaning.

### ✅ Lesson:
NER is not independent word classification.  
Context is necessary.

---

## 3️⃣ BiLSTM (Contextual Model)

Architecture:

Improvements:

- Sees left context
- Sees right context
- Learns contextual patterns

Example:

- "won the match" → PER likely
- "located in" → LOC likely

### ✅ Strength:
Better ambiguity resolution.

### ⚠️ Limitation:
Predictions still made independently per token.

It does not enforce valid BIO transitions.

---

## 4️⃣ BiLSTM + CRF (Structured Prediction)

Architecture:

CRF models:

- Label transition probabilities
- Global sequence score
- Valid BIO constraints

Score formulation:

[
Score(x,y) =
\sum_i emission_i +
\sum_i transition(y_{i-1}, y_i)
]

CRF decoding chooses:

[
y^* = \arg\max_y Score(x,y)
]

### ✅ Improvements:

- Prevents invalid sequences like:
  - O → I-ORG
- Improves multi-word entity consistency
- Better boundary detection

### ✅ Lesson:
NER is a structured prediction problem.

---

# 🔍 Ambiguity Case Study

Consider:

Why does performance differ?

| Model | Behavior |
|--------|----------|
| Token Classifier | Same prediction for "Jordan" |
| BiLSTM | Uses context words like "signed", "won" |
| BiLSTM+CRF | Also enforces valid entity boundaries |

Ambiguity resolution requires:

- Context modelling
- Sequence modelling
- Structured decoding

---

# 📈 Error Pattern Analysis

Common error types:

### 1️⃣ Boundary Errors
- Missing B- tag
- Incorrect I- tag continuation

### 2️⃣ Type Confusion
- PER vs LOC
- ORG vs O

### 3️⃣ Rare Entities
- Low-frequency names
- Limited training exposure

### 4️⃣ Long Entities
- University of California Berkeley
- Bank of America Corporation

CRF particularly helps with boundary consistency.

---

# 🧩 Theoretical Reflection

NER models differ in how they approximate:

[
P(y \mid x)
]

| Model | Approximation Strategy |
|--------|-----------------------|
| Token | P(y_i \| x_i) |
| BiLSTM | P(y_i \| x_{1:n}) |
| BiLSTM+CRF | Global sequence P(y \| x) |

This explains performance hierarchy.

---

# 🎓 Key Conceptual Insights

✅ NER requires context  
✅ Sequence modelling improves performance  
✅ Structured prediction improves consistency  
✅ Evaluation must be entity-level  
✅ Ambiguity is the central challenge  

---

# 🧠 Reflection Questions

1️⃣ Why is token accuracy misleading in NER?

2️⃣ Why does BiLSTM outperform token classifier?

3️⃣ What specific errors does CRF correct?

4️⃣ Would CRF still help if context modelling were weak?

5️⃣ How would performance change with:
- Larger dataset?
- More entity types?
- Nested entities?

---

# 🌟 Final Takeaway

NER is not:

> Word classification

It is:

> Structured sequence modelling under ambiguity

The progression from:

Token → BiLSTM → BiLSTM+CRF

demonstrates how:

- Context
- Structure
- Global optimization

incrementally improve performance.

---

# ✅ Conclusion

Understanding *why* models improve is more important than memorizing F1 scores.

Today’s key lesson:

> Ambiguity requires structure-aware learning.

🚀 In advanced systems, this idea extends further to:

- Transformer models (BERT)
- Span-based NER
- Entity Linking (NED)
- Joint structured prediction

We now have a solid foundation for those topics.